# Export the fine-tuned PP-OCRv5 fa/ckb recogniser to ONNX

The training run (`finetune_ppocr_fa_ckb.ipynb`, kernel `kyc-ppocr-fa-ckb`) hit
Kaggle's 12-hour limit in epoch 10, after saving `output/best_accuracy.pdparams`
but before its export cells ran. This notebook runs only those cells: it takes
the checkpoint (downloaded from that run and uploaded as a private dataset), exports base and fine-tuned models to
ONNX and scores both on the validation lines exactly like the training notebook.

**Inputs:** dataset `kyc-ppocr-fa-ckb-ckpt` (that run's `output/`), dataset `kyc-ocr-synth-fa-ckb`.
CPU is enough. **Output:** `ppocr_fa_ckb_rec.onnx`, `results.json`.


In [ ]:
# 1. Settings
EPOCHS = 12            # 196k lines / 48 = ~4k steps per epoch
LR = 2e-4              # fine-tune: under the 5e-4 the base was trained with
BATCH = 48             # 128 ran out of memory on a 16 GB T4 at the first step
PADDLEOCR_BRANCH = "release/3.7"
PADDLE_VERSIONS = ["3.2.0", "3.1.1", "3.0.0"]   # first one that installs and sees the GPU
PRETRAINED_URL = ("https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/"
                  "arabic_PP-OCRv5_mobile_rec_pretrained.pdparams")

import glob, os, sys, json, re, subprocess, shutil
WORK = "/kaggle/working"
hits = glob.glob("/kaggle/input/**/train.txt", recursive=True)
assert hits, "dataset not found: add kyc-ocr-synth-fa-ckb to the notebook"
DATA = os.path.dirname(hits[0])
print("data:", DATA, "| images:", len(os.listdir(os.path.join(DATA, "images"))))
CKPT = glob.glob("/kaggle/input/**/best_accuracy.pdparams", recursive=True)
if not CKPT:  # a dataset made from a zipped folder can arrive still zipped
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        shutil.unpack_archive(z, f"{WORK}/ckpt")
    CKPT = glob.glob(f"{WORK}/ckpt/**/best_accuracy.pdparams", recursive=True)
print([p for p in glob.glob("/kaggle/input/**", recursive=True) if "ckpt" in p][:10])
assert CKPT, "checkpoint not found: add the kyc-ppocr-fa-ckb-ckpt dataset"
CKPT = CKPT[0][: -len(".pdparams")]
print("checkpoint:", CKPT, os.path.getsize(CKPT + ".pdparams") // 2**20, "MB")


In [ ]:
# 2. PaddlePaddle (CPU is enough to export) and PaddleOCR
def sh(cmd):
    print("$", cmd)
    return subprocess.run(cmd, shell=True).returncode

done = False
for v in PADDLE_VERSIONS:
    if sh(f"pip install -q paddlepaddle=={v} -i https://www.paddlepaddle.org.cn/packages/stable/cpu/") == 0             and subprocess.run([sys.executable, "-c", "import paddle; print(paddle.__version__)"]).returncode == 0:
        print("using paddlepaddle", v); done = True; break
assert done, "no paddlepaddle build installed"

PADDLEOCR = f"{WORK}/PaddleOCR"
if not os.path.exists(PADDLEOCR):
    sh(f"git clone -q --depth 1 -b {PADDLEOCR_BRANCH} https://github.com/PaddlePaddle/PaddleOCR {PADDLEOCR}")
sh(f"pip install -q -r {PADDLEOCR}/requirements.txt")
sh("pip install -q onnx onnxruntime")
# paddle2onnx 2.x has wheels up to Python 3.12 only, and Kaggle runs 3.13 (its
# onnx pin then builds from source and fails): convert in a 3.12 venv.
P2O_ENV = f"{WORK}/p2o"
assert sh("pip install -q uv") == 0 and sh(f"uv venv -q -p 3.12 {P2O_ENV}") == 0
assert sh(f"uv pip install -q -p {P2O_ENV}/bin/python paddlepaddle=={v} paddle2onnx==2.1.0 packaging setuptools") == 0
P2O = f"{P2O_ENV}/bin/paddle2onnx"
sh(f"git -C {PADDLEOCR} log -1 --format=%H")


In [ ]:
# 3. Config and character list
import yaml
os.chdir(PADDLEOCR)
BASE_CFG = "configs/rec/PP-OCRv5/multi_language/arabic_PP-OCRv5_mobile_rec.yaml"
cfg = yaml.safe_load(open(BASE_CFG, encoding="utf-8"))
DICT = os.path.abspath(cfg["Global"]["character_dict_path"])
chars = open(DICT, encoding="utf-8").read().splitlines()
ours = open(os.path.join(DATA, "charset_current.txt"), encoding="utf-8").read().splitlines()
assert chars == ours, "PaddleOCR's dict differs from the one in the service's model: stop and check"
print(len(chars), "characters, same as the model in service")
sorani = set("ئابپتجچحخدرڕزژسشعغفڤقکگلڵمنوۆهھەیێ")
persian = set("آابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی")
print("missing Persian:", persian - set(chars), "| missing Sorani:", sorani - set(chars))
MAX_LEN = cfg["Global"]["max_text_length"]

In [ ]:
# 4. Which way round do labels go?  Encode two letters and look.
sys.path.insert(0, PADDLEOCR)
from ppocr.data.imaug.label_ops import CTCLabelEncode
from ppocr.postprocess.rec_postprocess import CTCLabelDecode

enc = CTCLabelEncode(max_text_length=MAX_LEN, character_dict_path=DICT, use_space_char=True)
dec = CTCLabelDecode(character_dict_path=DICT, use_space_char=True)
idx = list(enc({"label": "اب"})["label"][:2])
a, b = enc.dict["ا"], enc.dict["ب"]
ENCODER_REVERSES = idx == [b, a]
assert idx in ([a, b], [b, a]), idx
print("encoder reverses labels:", ENCODER_REVERSES, "| decoder reverses output:", dec.reverse)

def to_visual(text):
    # reading order -> as drawn: reverse, but keep digit / Latin runs left to right
    out, run = [], []
    for ch in reversed(text):
        if (ch.isascii() and ch.isalnum()) or "٠" <= ch <= "٩" or "۰" <= ch <= "۹":
            run.append(ch); continue
        if run: out.extend(reversed(run)); run = []
        out.append(ch)
    out.extend(reversed(run))
    return "".join(out)

assert to_visual("شماره 1403") == "1403 " + "شماره"[::-1]
# The model must learn visual order. If the encoder already reverses, give it
# reading order; otherwise turn the labels round here.
fix = (lambda s: s) if ENCODER_REVERSES else to_visual

In [ ]:
# 5. Label files (visual order), dropping lines the model cannot hold
os.makedirs(f"{WORK}/data", exist_ok=True)
stats = {}
for split in ("train", "val"):
    kept, long_ = [], 0
    for line in open(os.path.join(DATA, f"{split}.txt"), encoding="utf-8"):
        path, label = line.rstrip("\n").split("\t", 1)
        if len(label) > MAX_LEN:
            long_ += 1; continue
        kept.append(f"{path}\t{fix(label)}")
    open(f"{WORK}/data/{split}.txt", "w", encoding="utf-8").write("\n".join(kept) + "\n")
    stats[split] = (len(kept), long_)
print({k: f"{n} kept, {d} longer than {MAX_LEN}" for k, (n, d) in stats.items()})

In [ ]:
# 6. Pretrained weights and the fine-tune config
PRE = f"{WORK}/pretrained/arabic_PP-OCRv5_mobile_rec_pretrained"
if not os.path.exists(PRE + ".pdparams"):
    os.makedirs(os.path.dirname(PRE), exist_ok=True)
    assert sh(f"wget -q -O {PRE}.pdparams {PRETRAINED_URL}") == 0
print(os.path.getsize(PRE + ".pdparams") // 2**20, "MB")

OUT = f"{WORK}/output"
g = cfg["Global"]
g.update(pretrained_model=PRE, save_model_dir=OUT, epoch_num=EPOCHS, eval_batch_step=[0, 4000],
         save_epoch_step=EPOCHS, print_batch_step=100, distributed=False, character_dict_path=DICT,
         save_res_path=f"{OUT}/predicts.txt", use_gpu=False)
cfg["Optimizer"]["lr"].update(learning_rate=LR, warmup_epoch=1)
for part, split in (("Train", "train"), ("Eval", "val")):
    cfg[part]["dataset"].update(data_dir=DATA, label_file_list=[f"{WORK}/data/{split}.txt"])
    cfg[part]["loader"]["num_workers"] = 4
cfg["Train"]["sampler"]["first_bs"] = BATCH
cfg["Train"]["loader"]["batch_size_per_card"] = BATCH
CFG = f"{WORK}/finetune.yml"
yaml.safe_dump(cfg, open(CFG, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(open(CFG, encoding="utf-8").read()[:600])

In [ ]:
# 9. Export base and fine-tuned to ONNX, with the character list in the metadata
import onnx

def export(weights, name):
    inf = f"{WORK}/inf_{name}"
    assert sh(f"python tools/export_model.py -c {CFG} -o Global.pretrained_model={weights} "
              f"Global.save_inference_dir={inf}") == 0
    model_file = "inference.json" if os.path.exists(f"{inf}/inference.json") else "inference.pdmodel"
    onnx_path = f"{WORK}/{name}.onnx"
    assert sh(f"{P2O} --model_dir {inf} --model_filename {model_file} "
              f"--params_filename inference.pdiparams --save_file {onnx_path} --opset_version 14") == 0
    model = onnx.load(onnx_path)
    del model.metadata_props[:]
    model.metadata_props.add(key="character", value="\n".join(chars))
    onnx.save(model, onnx_path)
    return onnx_path

BASE_ONNX = export(PRE, "base")
FT_ONNX = export(CKPT, "ppocr_fa_ckb_rec")
print(BASE_ONNX, FT_ONNX)

In [ ]:
# 10. Both ONNX models on the validation lines, the way the service runs them
# (kyc/modules/ocr/recognizers/ppocr_onnx.py: height 48, width >= 320, greedy CTC).
import cv2, math, numpy as np, onnxruntime as ort

def preprocess(img):
    h, w = img.shape[:2]
    ratio = w / max(h, 1)
    width = max(320, int(48 * ratio))
    resized = cv2.resize(img, (max(1, min(width, int(math.ceil(48 * ratio)))), 48)).astype(np.float32)
    resized = (resized.transpose(2, 0, 1) / 255.0 - 0.5) / 0.5
    blob = np.zeros((3, 48, width), np.float32)
    blob[:, :, : resized.shape[2]] = resized
    return blob[None]

def reader(path):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    charset = ["<blank>", *sess.get_modelmeta().custom_metadata_map["character"].splitlines(), " "]
    name = sess.get_inputs()[0].name
    def read(img):
        probs = sess.run(None, {name: preprocess(img)})[0][0]
        assert probs.shape[1] == len(charset), (probs.shape, len(charset))
        best = probs.argmax(1)
        keep = np.r_[True, best[1:] != best[:-1]] & (best != 0)
        return to_visual("".join(charset[i] for i in best[keep])).strip()  # visual -> reading
    return read

SORANI_ONLY = set("ەێڕڵۆھڤ")
def group(label):
    if set(label) & SORANI_ONLY: return "ckb"
    if set(label) & set("گژپچ"): return "fa_gzpc"
    return "fa"

val = [l.rstrip("\n").split("\t", 1) for l in open(os.path.join(DATA, "val.txt"), encoding="utf-8")]
# "letters": compared as the service compares names (kyc/modules/ocr/normalize.py
# + lexicon.match_key): Arabic ي ك ة ھ ... folded to Persian, marks, spaces and
# ZWNJ dropped. "exact": the raw string.
FOLD = str.maketrans({"ي": "ی", "ى": "ی", "ے": "ی", "ك": "ک", "ڪ": "ک", "ة": "ه", "ۀ": "ه",
                      "أ": "ا", "إ": "ا", "ٱ": "ا", "ھ": "ه", "ؤ": "و", " ": None, "\u200c": None})
MARKS = re.compile("[\u064b-\u0670\u0640\u06d6-\u06ed]")
def key(s): return MARKS.sub("", s.translate(FOLD))

results = {}
for name, path in (("base", BASE_ONNX), ("fine_tuned", FT_ONNX)):
    read, tally = reader(path), {}
    for rel, label in val:
        got = read(cv2.imread(os.path.join(DATA, rel)))
        t = tally.setdefault(group(label), [0, 0, 0])
        t[0] += key(got) == key(label); t[1] += got == label; t[2] += 1
    results[name] = {k: f"letters {a / n:.1%}  exact {e / n:.1%}  (n={n})" for k, (a, e, n) in sorted(tally.items())}
    print(name, json.dumps(results[name], ensure_ascii=False, indent=1))
# width check: the exported input must take any width
s = ort.InferenceSession(FT_ONNX, providers=["CPUExecutionProvider"])
print("input:", s.get_inputs()[0].shape)
for w in (320, 640):
    s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, 48, w), np.float32)})

In [ ]:
# 11. Keep only what is needed in the output
import hashlib
results.update(checkpoint="best_accuracy (epoch 10 of 12; the run hit the 12 h limit)", lr=LR,
               sha256=hashlib.sha256(open(FT_ONNX, "rb").read()).hexdigest())
json.dump(results, open(f"{WORK}/results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
os.chdir(WORK)
for d in ("PaddleOCR", "pretrained", "inf_base", "inf_ppocr_fa_ckb_rec", "data", "base.onnx", "output", "p2o", "ckpt"):
    p = f"{WORK}/{d}"
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))
for f in glob.glob(f"{OUT}/*"):
    if not os.path.basename(f).startswith("best_accuracy"):
        shutil.rmtree(f, ignore_errors=True) if os.path.isdir(f) else os.remove(f)
print(json.dumps(results, ensure_ascii=False, indent=2))
!ls -la /kaggle/working